# Experiment Orchestrator - Classifier Selection

Este notebook orquesta el experimento completo de selección de clasificador CNN.

**Backbones a evaluar:** ResNet-18, EfficientNet-B0, DenseNet-121

**Método:** Few-shot (N=50, N=100) con mismo sampling para todos

**Dimensiones:** Few-shot, Grad-CAM, Computacional

## Celda 1: Setup

Instala dependencias e importa módulos.

In [ ]:
# Instalar dependencias
# !pip install torch torchvision timm scikit-learn scipy pillow numpy pyyaml matplotlib seaborn

import sys
sys.path.insert(0, '..')

from modules import DataModule, CNNClassifier
from scripts import (
    convert_refuge_format,
    evaluate_classification,
    evaluate_gradcam_dataset,
    train_few_shot,
    run_benchmark
)

import yaml
import torch

# Cargar configuración
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Celda 2: Convert Data

Convierte el dataset REFUGE al formato del proyecto.

In [ ]:
# Convertir REFUGE → annotations.json + splits.json
import subprocess
subprocess.run(['python', '../scripts/convert_refuge_format.py'], check=True)

## Celda 3: Initialize DataModule

Carga los datos usando la configuración.

In [ ]:
# Inicializar DataModule
data_module = DataModule(config['data'])
train_loader = data_module.get_train_loader()
val_loader = data_module.get_val_loader()
test_loader = data_module.get_test_loader()

print(f'Train: {len(train_loader.dataset)} samples')
print(f'Val: {len(val_loader.dataset)} samples')
print(f'Test: {len(test_loader.dataset)} samples')

## Celda 4: Run Few-Shot Experiment

Para cadabackbone: entrena con N=50 y N=100, evalúa en val.

**ADVERTENCIA:** Esta celda puede tomar varias horas.

In [ ]:
results = {}

for backbone in config['backbones']:
    print(f'\n{"="*60}')
    print(f'Running few-shot experiment for: {backbone}')
    print(f'{"="*60}')
    
    # Few-shot training N=50 y N=100
    print('\n[1/4] Few-shot training...')
    few_shot_results = train_few_shot(backbone, data_module, config)
    
    # Evaluate Grad-CAM con mejor modelo (N=100)
    print('\n[2/4] Evaluating Grad-CAM...')
    model = load_best_model(backbone)  # Cargar mejor modelo de few-shot
    gradcam_metrics = evaluate_gradcam_dataset(model, val_loader, device)
    
    # Benchmark de inferencia
    print('\n[3/4] Running inference benchmark...')
    benchmark_results = run_benchmark(backbone, model, val_loader, device)
    
    # Compile results
    results[backbone] = {
        'few_shot': few_shot_results,
        'gradcam': gradcam_metrics,
        'computational': benchmark_results
    }
    
    # Save checkpoint
    save_checkpoint(backbone, results[backbone])

## Celda 5: Compile Results

Genera selection_summary.json y selection_report.md.

In [ ]:
# Calculate scores and determine winner
# Score = 0.40 × Mean(F1@N50, F1@N100) + 0.40 × IoU_GradCAM + 0.20 × (1 - VRAM_norm)

scores = {}
for backbone, res in results.items():
    f1_n50 = res['few_shot']['N50']['f1']
    f1_n100 = res['few_shot']['N100']['f1']
    f1_mean = (f1_n50 + f1_n100) / 2
    iou = res['gradcam']['mean_iou']
    vram = res['computational']['vram_batch_1_mb']
    
    # Normalize VRAM
    all_vrams = [r['computational']['vram_batch_1_mb'] for r in results.values()]
    vram_norm = (vram - min(all_vrams)) / (max(all_vrams) - min(all_vrams) + 1e-8)
    
    score = 0.40 * f1_mean + 0.40 * iou + 0.20 * (1 - vram_norm)
    scores[backbone] = score
    res['score'] = score

winner = max(scores, key=scores.get)
print(f'\nWinner: {winner} (score={scores[winner]:.4f})')

# Save summary
save_selection_summary(results, winner, config)

# Generate report
generate_selection_report(results, winner, config)